# framework_orchestrator\nGenerated from 05_orchestration/framework_orchestrator.py for Databricks notebook import.\n

In [ ]:
from __future__ import annotations\n\nimport argparse\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport json\nimport sys\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nCURRENT_DIR = Path(__file__).resolve().parent\nCOMMON_DIR = CURRENT_DIR.parent / "00_common"\nINGESTION_DIR = CURRENT_DIR.parent / "01_ingestion"\nPROCESSING_DIR = CURRENT_DIR.parent / "02_processing"\nPUBLISH_DIR = CURRENT_DIR.parent / "03_publish"\nAUDIT_DIR = CURRENT_DIR.parent / "04_audit"\n\nfor p in [COMMON_DIR, INGESTION_DIR, PROCESSING_DIR, PUBLISH_DIR, AUDIT_DIR]:\n    sys.path.append(str(p))\n\nfrom global_config import load_global_config\nfrom config_loader import (\n    active_sources,\n    dq_rules_for_entity,\n    mappings_for_entity,\n    parse_json_cell,\n    parse_json_list,\n    publish_rule_for_entity,\n    get_lifecycle_log_table,\n)\nfrom utils import get_logger, new_load_id, truncate_str\n\n_logger = get_logger("ingestion_framework")\nfrom ingest_api import ingest_api\nfrom ingest_file_autoloader import ingest_file_stream, write_file_stream_to_landing\nfrom ingest_file_batch import ingest_file_batch\nfrom ingest_jdbc import ingest_jdbc_batch\nfrom landing_engine import add_landing_metadata, write_landing\nfrom conformance_engine import apply_column_mappings, write_conformance\nfrom bronze_dq_engine import run_bronze_dq_checks\nfrom dq_engine import apply_dq_rules, split_valid_reject\nfrom publish_silver import apply_post_publish_actions, publish_append, publish_merge\nfrom audit_logger import (\n    write_bronze_dq_results,\n    write_dq_rule_results,\n    write_pipeline_audit,\n    write_reject_rows,\n    write_validation_events,\n    write_lifecycle_event,\n)\n\n\ndef _require_source_fields(source: dict) -> None:\n    required = [\n        "source_system",\n        "source_entity",\n        "source_type",\n        "landing_table",\n        "conformance_table",\n        "silver_table",\n        "publish_mode",\n    ]\n    missing = [k for k in required if not source.get(k)]\n    if missing:\n        raise ValueError(f"Invalid source config for {source}: missing {missing}")\n\n    for layer in ("landing", "conformance", "silver"):\n        table_type = str(source.get(f"{layer}_table_type", "managed")).strip().lower() or "managed"\n        if table_type not in {"managed", "external"}:\n            raise ValueError(\n                f"Invalid {layer}_table_type={table_type!r} for {source.get('source_system')}.{source.get('source_entity')}. "\n                "Supported values: managed|external"\n            )\n        table_path = str(source.get(f"{layer}_table_path", "")).strip()\n        if table_type == "external" and not table_path:\n            raise ValueError(\n                f"Missing {layer}_table_path for external table in "\n                f"{source.get('source_system')}.{source.get('source_entity')}"\n            )\n\n\ndef _layer_table_target(source: dict, layer: str) -> tuple[str, str | None]:\n    table_type = str(source.get(f"{layer}_table_type", "managed")).strip().lower() or "managed"\n    table_path = str(source.get(f"{layer}_table_path", "")).strip() or None\n    return table_type, table_path\n\n\ndef _connection_profiles(global_config: dict) -> tuple[dict, dict]:\n    connections = global_config.get("connections", {})\n    return connections.get("jdbc_profiles", {}), connections.get("api_profiles", {})\n\n\ndef _ingest_source(spark, source: dict, global_config: dict):\n    source_type = source["source_type"].upper()\n    source_options = parse_json_cell(source.get("source_options_json"))\n    load_type = str(source.get("load_type", global_config.get("execution", {}).get("default_load_type", "incremental"))).lower()\n    jdbc_profiles, api_profiles = _connection_profiles(global_config)\n\n    if source_type == "JDBC":\n        profile_name = source.get("jdbc_profile", "")\n        jdbc_profile = jdbc_profiles.get(profile_name, {})\n        if load_type == "incremental" and source.get("watermark_column") and source_options.get("incremental_start_value"):\n            watermark_col = source["watermark_column"]\n            start_value = str(source_options["incremental_start_value"]).replace("'", "''")\n            table = source.get("jdbc_table", "")\n            source = {\n                **source,\n                "jdbc_table": f"(SELECT * FROM {table} WHERE {watermark_col} > '{start_value}') t",\n            }\n        return ingest_jdbc_batch(spark=spark, jdbc_profile=jdbc_profile, source=source, extra_options=source_options)\n\n    if source_type == "API":\n        profile_name = source.get("api_profile", "")\n        api_profile = api_profiles.get(profile_name, {})\n        query_params = parse_json_cell(source.get("api_query_params_json"))\n        if load_type == "incremental" and source.get("watermark_column") and source_options.get("incremental_start_value"):\n            query_params[source["watermark_column"]] = source_options["incremental_start_value"]\n        # Pass source_options so pagination config (pagination_type, page_size, etc.)\n        # is read from source_options_json, not from the flat registry row.\n        records = ingest_api(source=source, api_profile=api_profile, params=query_params, source_options=source_options)\n        return spark.createDataFrame(records)\n\n    if source_type == "FILE":\n        file_mode = str(source_options.get("file_ingest_mode", "batch")).lower()\n        if file_mode == "autoloader":\n            return ingest_file_stream(spark=spark, source=source, global_config=global_config, source_options=source_options)\n        return ingest_file_batch(spark=spark, source=source, global_config=global_config, source_options=source_options)\n\n    raise ValueError(f"Unsupported source_type: {source_type}")\n\n\ndef _audit_table_name(global_config: dict) -> str:\n    table_name = global_config.get("audit", {}).get("pipeline_runs_table", "")\n    if not table_name:\n        raise ValueError("audit.pipeline_runs_table must be configured in global_config.yaml")\n    return table_name\n\n\ndef _execution_config(global_config: dict) -> dict:\n    return global_config.get("execution", {})\n\n\ndef _retry_policy(global_config: dict, source_type: str) -> tuple[int, float]:\n    execution = _execution_config(global_config)\n    per_source_type = execution.get("retry_by_source_type", {})\n    source_policy = per_source_type.get(source_type.lower(), {})\n\n    max_retries = int(source_policy.get("max_retries", execution.get("max_retries", 1)))\n    backoff_seconds = float(source_policy.get("backoff_seconds", execution.get("retry_backoff_seconds", 2)))\n    return max_retries, backoff_seconds\n\n\ndef _run_with_retry(func, max_retries: int, backoff_seconds: float, stage_name: str):\n    last_error: Exception | None = None\n    for attempt in range(max_retries + 1):\n        try:\n            return func()\n        except Exception as exc:  # pragma: no cover\n            last_error = exc\n            if attempt >= max_retries:\n                break\n            time.sleep(backoff_seconds * (attempt + 1))\n    if last_error is None:\n        raise RuntimeError(f"{stage_name} failed with unknown error")\n    raise RuntimeError(f"{stage_name} failed after {max_retries + 1} attempt(s): {last_error}") from last_error\n\n\ndef _require_runtime_config(global_config: dict) -> None:\n    required = [\n        ("organization.framework_name", global_config.get("organization", {}).get("framework_name")),\n        ("audit.pipeline_runs_table", global_config.get("audit", {}).get("pipeline_runs_table")),\n    ]\n    missing = [name for name, value in required if not value]\n    if missing:\n        raise ValueError(f"Missing required global config values: {missing}")\n\n\ndef _publish_rule_lists(publish_rule: dict | None) -> tuple[list[str], list[str]]:\n    if not publish_rule:\n        return [], []\n    partition_columns = [str(c) for c in parse_json_list(publish_rule.get("partition_columns_json")) if str(c).strip()]\n    zorder_columns = [str(c) for c in parse_json_list(publish_rule.get("optimize_zorder_json")) if str(c).strip()]\n    return partition_columns, zorder_columns\n\n\ndef _log_event(event: str, **fields) -> None:\n    payload = {\n        "ts": datetime.now(timezone.utc).isoformat(),\n        "event": event,\n        **fields,\n    }\n    _logger.info(json.dumps(payload))\n\n\ndef _append_validation_event(events: list[dict], event: str, status: str = "INFO", **payload) -> None:\n    events.append(\n        {\n            "event": event,\n            "status": status,\n            "payload": payload,\n        }\n    )\n\n\ndef process_source(\n    config_dir: str,\n    source: dict,\n    global_config: dict,\n    spark=None,\n    dry_run: bool = True,\n    pk_check_summary: bool = False,\n) -> dict:\n\n    _require_source_fields(source)\n    _require_runtime_config(global_config)\n\n    # Lifecycle log setup\n    lifecycle_log_table = get_lifecycle_log_table(global_config)\n    user = global_config.get("run_user", "system")\n    entity_id = f"{source.get('source_system','')}.{source.get('source_entity','')}"\n    # Log onboarding event\n    event_type = "onboarded"\n    event_details = json.dumps({k: source.get(k) for k in ['source_type', 'landing_table', 'conformance_table', 'silver_table']})\n    if spark is not None:\n        write_lifecycle_event(spark, lifecycle_log_table, entity_id, event_type, event_details, user)\n\n    source_system = source["source_system"]\n    source_entity = source["source_entity"]\n    source_type = source["source_type"].upper()\n    load_id = new_load_id()\n    stage_seconds: dict[str, float] = {}\n    validation_events: list[dict] = []\n\n    _log_event(\n        "source_start",\n        source=f"{source_system}.{source_entity}",\n        load_id=load_id,\n        source_type=source_type,\n        dry_run=dry_run,\n    )\n    _append_validation_event(\n        validation_events,\n        "source_start",\n        source=f"{source_system}.{source_entity}",\n        load_id=load_id,\n        source_type=source_type,\n        dry_run=dry_run,\n    )\n\n    mapping_rows = mappings_for_entity(\n        config_dir,\n        source_system,\n        source_entity,\n        product_name=source.get("product_name"),\n        spark=spark,\n        global_config=global_config,\n    )\n    dq_rows = dq_rules_for_entity(\n        config_dir,\n        source_system,\n        source_entity,\n        product_name=source.get("product_name"),\n        spark=spark,\n        global_config=global_config,\n    )\n    publish_rule = publish_rule_for_entity(\n        config_dir,\n        source_system,\n        source_entity,\n        product_name=source.get("product_name"),\n        spark=spark,\n        global_config=global_config,\n    )\n\n    _log_event(\n        "metadata_resolved",\n        source=f"{source_system}.{source_entity}",\n        mapping_count=len(mapping_rows),\n        dq_rule_count=len(dq_rows),\n        publish_rule_found=bool(publish_rule),\n    )\n    _append_validation_event(\n        validation_events,\n        "metadata_resolved",\n        status="PASS",\n        mapping_count=len(mapping_rows),\n        dq_rule_count=len(dq_rows),\n        publish_rule_found=bool(publish_rule),\n    )\n\n    if dry_run or spark is None:\n        pk_cols = [c.strip() for c in str(source.get("primary_key", "")).split(",") if c.strip()]\n        pk_summary = {\n            "primary_key_configured": bool(pk_cols),\n            "primary_key_columns": pk_cols,\n            "primary_key_checks": ["primary_key_not_null", "primary_key_unique"] if pk_cols else [],\n            "primary_key_failure_counts": None,\n        }\n        _log_event(\n            "verification_dry_run",\n            source=f"{source_system}.{source_entity}",\n            checks={\n                "routing_metadata_resolved": True,\n                "mappings_loaded": len(mapping_rows),\n                "dq_rules_loaded": len(dq_rows),\n                "publish_mode_configured": bool(source.get("publish_mode")),\n                **({"pk_summary": pk_summary} if pk_check_summary else {}),\n            },\n        )\n        _append_validation_event(\n            validation_events,\n            "verification_dry_run",\n            status="PASS",\n            routing_metadata_resolved=True,\n            mappings_loaded=len(mapping_rows),\n            dq_rules_loaded=len(dq_rows),\n            publish_mode_configured=bool(source.get("publish_mode")),\n        )\n        return {\n            "tenant": source.get("tenant", ""),\n            "brand": source.get("brand", ""),\n            "product_name": source.get("product_name", ""),\n            "source": f"{source_system}.{source_entity}",\n            "mode": "dry_run",\n            "steps": [\n                f"ingest via {source['source_type']}",\n                f"write landing {source['landing_table']}",\n                f"apply {len(mapping_rows)} mappings",\n                f"apply {len(dq_rows)} dq rules",\n                f"publish {source['publish_mode']} to {source['silver_table']}",\n            ],\n            **({"pk_check_summary": pk_summary} if pk_check_summary else {}),\n        }\n    fail_fast = bool(_execution_config(global_config).get("fail_fast", True))\n    max_retries, backoff_seconds = _retry_policy(global_config, source_type=source_type)\n    audit_cfg = global_config.get("audit", {})\n    framework_name = global_config.get("organization", {}).get("framework_name", "")\n    audit_table = _audit_table_name(global_config)\n\n    try:\n        t0 = time.monotonic()\n        # Log update event (example: after metadata resolved)\n        event_type = "metadata_resolved"\n        event_details = json.dumps({\n            "mapping_count": len(mapping_rows),\n            "dq_rule_count": len(dq_rows),\n            "publish_rule_found": bool(publish_rule)\n        })\n        if spark is not None:\n            write_lifecycle_event(spark, lifecycle_log_table, entity_id, event_type, event_details, user)\n        raw_df = _run_with_retry(\n            lambda: _ingest_source(spark, source, global_config),\n            max_retries=max_retries,\n            backoff_seconds=backoff_seconds,\n            stage_name=f"ingest:{source_system}.{source_entity}",\n        )\n        stage_seconds["ingest"] = round(time.monotonic() - t0, 3)\n        _log_event(\n            "stage_complete",\n            source=f"{source_system}.{source_entity}",\n            stage="ingestion_adapter",\n            seconds=stage_seconds["ingest"],\n            verification={"dataframe_created": raw_df is not None},\n        )\n        _append_validation_event(\n            validation_events,\n            "ingestion_adapter",\n            status="PASS" if raw_df is not None else "FAIL",\n            seconds=stage_seconds["ingest"],\n            dataframe_created=raw_df is not None,\n        )\n\n        if getattr(raw_df, "isStreaming", False):\n            source_options = parse_json_cell(source.get("source_options_json"))\n            t1 = time.monotonic()\n            streaming_landing_df = add_landing_metadata(\n                raw_df,\n                source_system=source_system,\n                source_entity=source_entity,\n                load_id=load_id,\n            )\n            write_file_stream_to_landing(\n                spark=spark,\n                streaming_df=streaming_landing_df,\n                source=source,\n                global_config=global_config,\n                source_options=source_options,\n            )\n            stage_seconds["landing_stream"] = round(time.monotonic() - t1, 3)\n            _log_event(\n                "stage_complete",\n                source=f"{source_system}.{source_entity}",\n                stage="landing_stream",\n                seconds=stage_seconds["landing_stream"],\n                verification={"streaming_query_completed": True},\n            )\n            _append_validation_event(\n                validation_events,\n                "landing_stream",\n                status="PASS",\n                seconds=stage_seconds["landing_stream"],\n                streaming_query_completed=True,\n            )\n\n            validation_events_table = audit_cfg.get("validation_events_table", "")\n            if validation_events_table:\n                write_validation_events(\n                    spark,\n                    validation_events_table,\n                    load_id,\n                    source_system,\n                    source_entity,\n                    validation_events,\n                )\n\n            write_pipeline_audit(\n                spark,\n                table_name=audit_table,\n                run_rows=[\n                    (\n                        framework_name,\n                        load_id,\n                        source_system,\n                        source_entity,\n                        0,\n                        0,\n                        "SUCCESS",\n                        None,\n                        None,\n                    )\n                ],\n            )\n\n            return {\n                "source": f"{source_system}.{source_entity}",\n                "mode": "execute_streaming",\n                "landing_table": source["landing_table"],\n                "status": "SUCCESS",\n                "stage_seconds": stage_seconds,\n            }\n\n        t2 = time.monotonic()\n        source_options = parse_json_cell(source.get("source_options_json"))\n        raw_columns = list(raw_df.columns)\n        raw_count = raw_df.count()\n        bronze_out = run_bronze_dq_checks(raw_df, source_options=source_options)\n        bronze_df = bronze_out["annotated_df"]\n        bronze_valid_df = bronze_out["valid_df"]\n        bronze_reject_df = bronze_out["reject_df"]\n        bronze_results = bronze_out["results"]\n        bronze_config_errors = bronze_out["config_errors"]\n\n        # Cache counts once — reused in stage log, append_event, and verification_summary.\n        bronze_valid_count = bronze_valid_df.count()\n        bronze_reject_count = bronze_reject_df.count()\n\n        _log_event(\n            "stage_complete",\n            source=f"{source_system}.{source_entity}",\n            stage="bronze_dq_engine",\n            verification={\n                "config_errors": bronze_config_errors,\n                "checks": bronze_results,\n                "bronze_valid_rows": bronze_valid_count,\n                "bronze_reject_rows": bronze_reject_count,\n            },\n        )\n        _append_validation_event(\n            validation_events,\n            "bronze_dq_engine",\n            status="PASS" if not bronze_config_errors else "FAIL",\n            config_errors=bronze_config_errors,\n            checks=bronze_results,\n            bronze_valid_rows=bronze_valid_count,\n            bronze_reject_rows=bronze_reject_count,\n        )\n\n        landing_df = add_landing_metadata(bronze_df, source_system=source_system, source_entity=source_entity, load_id=load_id)\n        # Archive logic: get archive_table and retention_days from config or source_options_json\n        archive_table = None\n        retention_days = None\n        # Priority: source_options_json > global_config > None\n        if source_options.get("archive_table"):\n            archive_table = source_options["archive_table"]\n        elif global_config.get("archive", {}).get("archive_table"):\n            archive_table = global_config["archive"]["archive_table"]\n        if source_options.get("archive_retention_days"):\n            retention_days = int(source_options["archive_retention_days"])\n        elif global_config.get("archive", {}).get("retention_days"):\n            retention_days = int(global_config["archive"]["retention_days"])\n\n        landing_table_type, landing_table_path = _layer_table_target(source, "landing")\n        write_landing(\n            landing_df,\n            source["landing_table"],\n            table_type=landing_table_type,\n            external_path=landing_table_path,\n            archive_table=archive_table,\n            retention_days=retention_days,\n        )\n        stage_seconds["landing"] = round(time.monotonic() - t2, 3)\n        required_landing_cols = {"source_system", "source_entity", "load_id", "ingest_ts"}\n        landing_has_tech_cols = required_landing_cols.issubset(set(landing_df.columns))\n        _log_event(\n            "stage_complete",\n            source=f"{source_system}.{source_entity}",\n            stage="landing_raw",\n            seconds=stage_seconds["landing"],\n            verification={\n                "raw_row_count": raw_count,\n                "raw_columns_preserved": set(raw_columns).issubset(set(landing_df.columns)),\n                "landing_technical_columns_present": landing_has_tech_cols,\n            },\n        )\n        _append_validation_event(\n            validation_events,\n            "landing_raw",\n            status="PASS" if landing_has_tech_cols else "FAIL",\n            seconds=stage_seconds["landing"],\n            raw_row_count=raw_count,\n            raw_columns_preserved=set(raw_columns).issubset(set(landing_df.columns)),\n            landing_technical_columns_present=landing_has_tech_cols,\n        )\n\n        t3 = time.monotonic()\n        conformance_df = apply_column_mappings(bronze_valid_df, mapping_rows)\n        conformance_table_type, conformance_table_path = _layer_table_target(source, "conformance")\n        write_conformance(\n            conformance_df,\n            source["conformance_table"],\n            table_type=conformance_table_type,\n            external_path=conformance_table_path,\n        )\n        stage_seconds["conformance"] = round(time.monotonic() - t3, 3)\n        expected_conf_cols = [r.get("conformance_column", "") for r in mapping_rows if r.get("conformance_column")]\n        missing_conf_cols = [c for c in expected_conf_cols if c not in conformance_df.columns]\n        _log_event(\n            "stage_complete",\n            source=f"{source_system}.{source_entity}",\n            stage="conformance_engine",\n            seconds=stage_seconds["conformance"],\n            verification={\n                "expected_conformance_columns": expected_conf_cols,\n                "missing_conformance_columns": missing_conf_cols,\n                "conformance_columns_ok": len(missing_conf_cols) == 0,\n            },\n        )\n        _append_validation_event(\n            validation_events,\n            "conformance_engine",\n            status="PASS" if len(missing_conf_cols) == 0 else "FAIL",\n            seconds=stage_seconds["conformance"],\n            expected_conformance_columns=expected_conf_cols,\n            missing_conformance_columns=missing_conf_cols,\n        )\n\n        t4 = time.monotonic()\n        dq_df = apply_dq_rules(\n            conformance_df,\n            dq_rows,\n            primary_key=source.get("primary_key", ""),\n        )\n        valid_df, reject_df = split_valid_reject(dq_df)\n\n        pk_cols = [c.strip() for c in str(source.get("primary_key", "")).split(",") if c.strip()]\n        pk_summary_exec: dict[str, object] = {\n            "primary_key_configured": bool(pk_cols),\n            "primary_key_columns": pk_cols,\n            "primary_key_checks": ["primary_key_not_null", "primary_key_unique"] if pk_cols else [],\n        }\n        if pk_cols:\n            pk_not_null_failures = dq_df.filter(\n                F.col("dq_failed_rule").contains("primary_key_not_null")\n            ).count()\n            pk_unique_failures = dq_df.filter(\n                F.col("dq_failed_rule").contains("primary_key_unique")\n            ).count()\n            pk_summary_exec["primary_key_failure_counts"] = {\n                "primary_key_not_null": pk_not_null_failures,\n                "primary_key_unique": pk_unique_failures,\n            }\n        else:\n            pk_summary_exec["primary_key_failure_counts"] = None\n        stage_seconds["dq"] = round(time.monotonic() - t4, 3)\n        _log_event(\n            "stage_complete",\n            source=f"{source_system}.{source_entity}",\n            stage="dq_engine",\n            seconds=stage_seconds["dq"],\n            verification={\n                "dq_columns_present": {"dq_status", "dq_failed_rule"}.issubset(set(dq_df.columns)),\n            },\n        )\n        _append_validation_event(\n            validation_events,\n            "dq_engine",\n            status="PASS" if {"dq_status", "dq_failed_rule"}.issubset(set(dq_df.columns)) else "FAIL",\n            seconds=stage_seconds["dq"],\n            dq_columns_present={"dq_status", "dq_failed_rule"}.issubset(set(dq_df.columns)),\n            **({"pk_summary": pk_summary_exec} if pk_check_summary else {}),\n        )\n\n        mode_val = (publish_rule or {}).get("publish_mode", source.get("publish_mode"))\n        mode = mode_val.lower() if mode_val else "append"\n        partition_columns, zorder_columns = _publish_rule_lists(publish_rule)\n        t5 = time.monotonic()\n        if mode == "append":\n            silver_table_type, silver_table_path = _layer_table_target(source, "silver")\n            _run_with_retry(\n                lambda: publish_append(\n                    valid_df,\n                    source["silver_table"],\n                    partition_columns=partition_columns,\n                    table_type=silver_table_type,\n                    external_path=silver_table_path,\n                ),\n                max_retries=max_retries,\n                backoff_seconds=backoff_seconds,\n                stage_name=f"publish_append:{source_system}.{source_entity}",\n            )\n        elif mode == "merge":\n            merge_key = (publish_rule or {}).get("merge_key") or source.get("primary_key", "")\n            if not merge_key:\n                raise ValueError(f"Missing merge key for {source_system}.{source_entity}")\n            _run_with_retry(\n                lambda: publish_merge(spark, valid_df, source["silver_table"], merge_key),\n                max_retries=max_retries,\n                backoff_seconds=backoff_seconds,\n                stage_name=f"publish_merge:{source_system}.{source_entity}",\n            )\n        else:\n            raise ValueError(f"Unsupported publish mode: {mode}")\n\n        post_publish_optimize = bool(_execution_config(global_config).get("post_publish_optimize", True))\n        if post_publish_optimize:\n            apply_post_publish_actions(spark, source["silver_table"], zorder_columns=zorder_columns)\n        stage_seconds["publish"] = round(time.monotonic() - t5, 3)\n\n        _log_event(\n            "stage_complete",\n            source=f"{source_system}.{source_entity}",\n            stage="silver_publish",\n            seconds=stage_seconds["publish"],\n            verification={\n                "publish_mode": mode,\n                "partition_columns": partition_columns,\n                "zorder_columns": zorder_columns,\n                "post_publish_optimize": post_publish_optimize,\n            },\n        )\n        _append_validation_event(\n            validation_events,\n            "silver_publish",\n            status="PASS",\n            seconds=stage_seconds["publish"],\n            publish_mode=mode,\n            partition_columns=partition_columns,\n            zorder_columns=zorder_columns,\n            post_publish_optimize=post_publish_optimize,\n        )\n\n        # Count once near the end to avoid repetitive expensive actions.\n        landing_count = landing_df.count()\n        conformance_count = conformance_df.count()\n        valid_count = valid_df.count()\n        reject_count = reject_df.count()\n\n        checks = {\n            "landing_equals_ingestion_for_batch": landing_count == raw_count,\n            "bronze_valid_plus_bronze_reject_equals_ingestion": (bronze_valid_count + bronze_reject_count) == raw_count,\n            "valid_plus_reject_equals_conformance": (valid_count + reject_count) == conformance_count,\n            "append_expected_silver_increase_equals_valid": mode != "append" or valid_count >= 0,\n        }\n        _log_event(\n            "verification_summary",\n            source=f"{source_system}.{source_entity}",\n            load_id=load_id,\n            counts={\n                "ingestion_rows": raw_count,\n                "landing_rows": landing_count,\n                "bronze_valid_rows": bronze_valid_count,\n                    "bronze_reject_rows": bronze_reject_count,\n                "conformance_rows": conformance_count,\n                "valid_rows": valid_count,\n                "reject_rows": reject_count,\n            },\n            checks=checks,\n        )\n        _append_validation_event(\n            validation_events,\n            "verification_summary",\n            status="PASS" if all(checks.values()) else "FAIL",\n            counts={\n                "ingestion_rows": raw_count,\n                "landing_rows": landing_count,\n                "bronze_valid_rows": bronze_valid_count,\n                "bronze_reject_rows": bronze_reject_count,\n                "conformance_rows": conformance_count,\n                "valid_rows": valid_count,\n                "reject_rows": reject_count,\n            },\n            checks=checks,\n        )\n\n        bronze_results_table = audit_cfg.get("bronze_dq_results_table", "")\n        if bronze_results_table:\n            write_bronze_dq_results(\n                spark,\n                bronze_results_table,\n                load_id,\n                source_system,\n                source_entity,\n                bronze_results,\n                config_errors=bronze_config_errors,\n            )\n\n        dq_results_table = audit_cfg.get("dq_results_table", "")\n        if dq_results_table:\n            write_dq_rule_results(spark, dq_results_table, load_id, source_system, source_entity, dq_df)\n\n        rejects_table = audit_cfg.get("rejects_table", "")\n        if rejects_table and reject_count > 0:\n            write_reject_rows(reject_df, rejects_table, load_id, source_system, source_entity)\n\n        validation_events_table = audit_cfg.get("validation_events_table", "")\n        if validation_events_table:\n            write_validation_events(\n                spark,\n                validation_events_table,\n                load_id,\n                source_system,\n                source_entity,\n                validation_events,\n            )\n\n        write_pipeline_audit(\n            spark,\n            table_name=audit_table,\n            run_rows=[\n                (\n                    framework_name,\n                    load_id,\n                    source_system,\n                    source_entity,\n                    landing_count,\n                    valid_count,\n                    "SUCCESS",\n                    None,\n                    None,\n                )\n            ],\n        )\n\n        return {\n            "source": f"{source_system}.{source_entity}",\n            "mode": "execute",\n            "status": "SUCCESS",\n            "valid_rows": valid_count,\n            "reject_rows": reject_count,\n            "publish_mode": mode,\n            "stage_seconds": stage_seconds,\n            **({"pk_check_summary": pk_summary_exec} if pk_check_summary else {}),\n        }\n    except Exception as exc:\n        # Log failure event\n        event_type = "failed"\n        event_details = truncate_str(str(exc))\n        if spark is not None:\n            write_lifecycle_event(spark, lifecycle_log_table, entity_id, event_type, event_details, user)\n        _log_event(\n            "source_failed",\n            source=f"{source_system}.{source_entity}",\n            load_id=load_id,\n            error=truncate_str(str(exc)),\n            stage_seconds=stage_seconds,\n        )\n        _append_validation_event(\n            validation_events,\n            "source_failed",\n            status="FAIL",\n            error=truncate_str(str(exc)),\n            stage_seconds=stage_seconds,\n        )\n        validation_events_table = audit_cfg.get("validation_events_table", "")\n        if validation_events_table:\n            write_validation_events(\n                spark,\n                validation_events_table,\n                load_id,\n                source_system,\n                source_entity,\n                validation_events,\n            )\n        write_pipeline_audit(\n            spark,\n            table_name=audit_table,\n            run_rows=[\n                (\n                    framework_name,\n                    load_id,\n                    source_system,\n                    source_entity,\n                    0,\n                    0,\n                    "FAILED",\n                    truncate_str(str(exc)),\n                    None,\n                )\n            ],\n        )\n        if fail_fast:\n            raise\n        return {\n            "source": f"{source_system}.{source_entity}",\n            "mode": "execute",\n            "status": "FAILED",\n            "error": truncate_str(str(exc)),\n            "stage_seconds": stage_seconds,\n        }\n\n\ndef run_all(\n    config_dir: str,\n    global_config: dict,\n    spark=None,\n    dry_run: bool = True,\n    product_name: str | None = None,\n    source_system: str | None = None,\n    source_entity: str | None = None,\n    parallel_workers: int = 0,\n    pk_check_summary: bool = False,\n) -> list[dict]:\n    sources = active_sources(\n        config_dir,\n        spark=spark,\n        global_config=global_config,\n        product_name=product_name,\n        source_system=source_system,\n        source_entity=source_entity,\n    )\n\n    if not sources:\n        all_active = active_sources(\n            config_dir,\n            spark=spark,\n            global_config=global_config,\n            product_name=None,\n            source_system=None,\n            source_entity=None,\n        )\n        available = sorted(\n            {\n                (\n                    str(s.get("product_name", "")).strip(),\n                    str(s.get("source_system", "")).strip(),\n                    str(s.get("source_entity", "")).strip(),\n                )\n                for s in all_active\n            }\n        )\n        _logger.warning(\n            "No active sources matched filters product_name=%r source_system=%r source_entity=%r. Available (product, system, entity): %s",\n            product_name,\n            source_system,\n            source_entity,\n            available,\n        )\n        return []\n\n    if parallel_workers <= 1:\n        results = []\n        for source in sources:\n            results.append(\n                process_source(\n                    config_dir=config_dir,\n                    source=source,\n                    global_config=global_config,\n                    spark=spark,\n                    dry_run=dry_run,\n                    pk_check_summary=pk_check_summary,\n                )\n            )\n        return results\n\n    _logger.info("Running %s sources in parallel with %s workers", len(sources), parallel_workers)\n    results = []\n    with ThreadPoolExecutor(max_workers=parallel_workers) as executor:\n        future_to_source = {\n            executor.submit(\n                process_source,\n                config_dir=config_dir,\n                source=source,\n                global_config=global_config,\n                spark=spark,\n                dry_run=dry_run,\n                pk_check_summary=pk_check_summary,\n            ): source\n            for source in sources\n        }\n        for future in as_completed(future_to_source):\n            source = future_to_source[future]\n            source_name = f"{source.get('source_system', '')}.{source.get('source_entity', '')}"\n            try:\n                results.append(future.result())\n            except Exception as exc:\n                _logger.exception("Parallel execution failed for %s: %s", source_name, exc)\n                raise\n    return results\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description="Run metadata-driven data ingestion orchestrator")\n    parser.add_argument("--config-dir", default=str(Path(__file__).resolve().parents[2] / "config"))\n    parser.add_argument("--global-config", default=str(Path(__file__).resolve().parents[2] / "config" / "global_config.yaml"))\n    parser.add_argument("--execute", action="store_true", help="Execute with Spark (Databricks runtime)")\n    parser.add_argument("--parallel", type=int, default=0, help="Run with N parallel workers (default 0 = sequential)")\n    parser.add_argument("--serial", action="store_true", help="Force sequential processing")\n    parser.add_argument("--pk-check-summary", action="store_true", help="Include primary key DQ summary in output")\n    parser.add_argument("--product-name", default=None, help="Optional product filter")\n    parser.add_argument("--source-system", default=None, help="Optional source system filter")\n    parser.add_argument("--source-entity", default=None, help="Optional source entity filter")\n    args = parser.parse_args()\n\n    global_config = load_global_config(args.global_config)\n    dry_run = not args.execute\n    requested_parallel = max(0, int(args.parallel or 0))\n    parallel_workers = 0 if args.serial else (requested_parallel if requested_parallel > 1 else 0)\n\n    results = run_all(\n        config_dir=args.config_dir,\n        global_config=global_config,\n        spark=globals().get("spark"),\n        dry_run=dry_run,\n        product_name=args.product_name,\n        source_system=args.source_system,\n        source_entity=args.source_entity,\n        parallel_workers=parallel_workers,\n        pk_check_summary=args.pk_check_summary,\n    )\n\n    if not results:\n        _logger.warning(\n            "Orchestrator returned no results. This usually means filter values do not match active metadata rows in config/source_registry.csv."\n        )\n    print(json.dumps(results, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n